# 02 - Data Manipulation with Pandas
Tento sešit demonstruje klíčové koncepty práce s tabulkovými daty pomocí knihovny Pandas:
- Načítání strukturovaných dat z formátu CSV relativní cesty
- Základní inspekce rozměrů, datových typů a názvů sloupců
- Indexování a slicing řádků pomocí `.iloc` a `.head()` (rozdíl mezi `Series` a `DataFrame`)
- Agregační statistiky sloupců (`mean`, `min`, `max`)
- Vektorizované operace a tvorba nových sloupců (`axis=1`)
- Práce s chybějícími daty (`isna().mean()`) a selektivní mazání sloupců (`.drop()`)

## 1. Import a načtení dat
Načtení lokálního CSV souboru z relativní složky `../data/`a ověření jeho struktury.

In [2]:
import pandas as pd 

#* Relativní cesta zajišťuje, že kód bude fungovat na jakémkoliv počítači po naklonování z GitHubu
df = pd.read_csv("../data/tech_products.csv")

#* Atributy tabulky
data_types = list(df.dtypes)
col_names = list(df.columns)
n_rows = df.shape[0]
n_cols = df.shape[1]

print(f"Rozměry tabulky: {n_rows} řádků a {n_cols} sloupců.")
print("Názvy sloupců:", col_names)
print("Datové typy:", data_types)

#? Zobrazení celé tabulky
df

Rozměry tabulky: 10 řádků a 8 sloupců.
Názvy sloupců: ['product_id', 'product_name', 'category', 'base_price', 'shipping_cost', 'warranty_fee', 'rating', 'stock_clearance']
Datové typy: [dtype('int64'), <StringDtype(storage='python', na_value=nan)>, <StringDtype(storage='python', na_value=nan)>, dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64'), dtype('float64')]


,product_id,product_name,category,base_price,shipping_cost,warranty_fee,rating,stock_clearance
0,1,Pro Ultra Laptop,Laptops,1200.0,15.0,50.0,4.8,NaN
1,2,Lite Slim Laptop,Laptops,650.0,15.0,30.0,4.2,NaN
2,3,Gamer Headset X,Audio,85.0,5.0,10.0,4.5,NaN
3,4,Wireless Earbuds,Audio,45.0,5.0,5.0,3.9,NaN
4,5,4K Studio Monitor,Displays,420.0,20.0,25.0,NaN,NaN
5,6,Curved Gaming Screen,Displays,310.0,20.0,20.0,4.6,NaN
6,7,Mechanical Keyboard,Peripherals,110.0,5.0,10.0,4.7,NaN
7,8,Ergonomic Mouse,Peripherals,60.0,5.0,5.0,NaN,NaN
8,9,USB-C Multi Dock,Peripherals,75.0,5.0,5.0,4.1,NaN
9,10,Noise Cancelling Headphones,Audio,230.0,8.0,15.0,4.9,NaN


## 2. Výběr řádků: `.head()` a `.iloc`
Ukázka rozdílu mezi 1D řezem (`pd.Series`) a 2D tabulkou (`pd.DataFrame`) při výběru přes `.iloc`.

In [ ]:

#* Náhled na prvních 5 řádků
fist_five = df.head(5)

#! Dvojité hranáté závorky [[1]] vrátí řádek jako 2D strukturu (pd.DataFrame)
second_row_df = df.iloc[[1]]

#! Jednoduchá závorka [-1] vrátí poslední řádek jako 1D strukturu (pd.Series)
last_row_series = df.iloc[-1]

#? Slicing [1:4] vybere řádky s indexy 1, 2 a 3 (horní mez 4 je exkluzivní)
subset_rows = df.iloc[1:4]

print("Typ se dvěma závorkami:", type(second_row_df))
print("Typ s jednou závorkou:", type(last_row_series))

# Zobrazení vybraného podvýběru
subset_rows

Typ se dvěma závorkami: <class 'pandas.DataFrame'>
Typ s jednou závorkou: <class 'pandas.Series'>


,product_id,product_name,category,base_price,shipping_cost,warranty_fee,rating,stock_clearance
1,2,Lite Slim Laptop,Laptops,650.0,15.0,30.0,4.2,NaN
2,3,Gamer Headset X,Audio,85.0,5.0,10.0,4.5,NaN
3,4,Wireless Earbuds,Audio,45.0,5.0,5.0,3.9,NaN


## 3. Agregační statistiky numerických sloupců
Výpočet základních deskriptivních metrik na sloupci cen (`base_price`).

In [ ]:

#* Jednotlivé agregační metody aplikované na číselný sloupec
price_mean = df["base_price"].mean()
price_max = df["base_price"].max()
price_min = df["base_price"].min()

print(f"Průměrná cena produktu: {price_mean: .2f} EUR")
print(f"Nejvyšší cena produktu: {price_max: .2f} EUR")
print(f"Nejnižší cena produktu: {price_min: .2f} EUR")

Průměrná cena produktu:  318.50 EUR
Nejvyšší cena produktu:  1200.00 EUR
Nejnižší cena produktu:  45.00 EUR


## 4. Tvorba nových příznaků (Feature Engineering)
Ukázka vektorizované matematické operace na jednom sloupci a horizontálního součtu napříč sloupci (`axis=1`) 

In [ ]:

#* Vektorizovaný výpočet: sleva 10 % z původní ceny (operace proběhne na všech řádcích bez cyklu for)
df["discounted_price"] = df["base_price"] * 0.90

#! 2. Horizontální součet napříč vybranými sloupci:
#! Parametr axis=1 sčítá hodnoty na každém řádku zleva doprava (base_price + shipping_cost + warranty_fee)
df["total_customer_price"] = df[["base_price", "shipping_cost", "warranty_fee"]].sum(axis=1)

#? Náhled na nově spočítané sloupce
df[["product_name", "base_price", "discounted_price", "total_customer_price"]].head()

,product_name,base_price,discounted_price,total_customer_price
0,Pro Ultra Laptop,1200.0,1080.0,1265.0
1,Lite Slim Laptop,650.0,585.0,695.0
2,Gamer Headset X,85.0,76.5,100.0
3,Wireless Earbuds,45.0,40.5,55.0
4,4K Studio Monitor,420.0,378.0,465.0


## 5. Detekce chybějících hodnot a čištění tabulky
Zjištění procentuálního zastoupení prázdných buněk a odstranění irelevantních sloupců metodou `.drop()`

In [9]:

#* isna() vrací masku True/False, .mean() spočítá poměr chybějících hodnot (True = 1, False = 0)
na_percentage = df.isna().mean()
print("Procento chybějících hodnot v jednotlivých sloupcích:")
print((na_percentage * 100).round(1).astype(str) + " %")

#! Odstranění prádzného nebo nepotřebného sloupce 'stock_clearance'
#! Parametr columns explicitně určuje, že mažeme sloupec a nikoliv řádek
df_cleaned = df.drop(columns=["stock_clearance"])

print("\nPůvodní počet sloupců:", df.shape[1])
print("Počet sloupců po očištění:", df_cleaned.shape[1])

Procento chybějících hodnot v jednotlivých sloupcích:
product_id                0.0 %
product_name              0.0 %
category                  0.0 %
base_price                0.0 %
shipping_cost             0.0 %
warranty_fee              0.0 %
rating                   20.0 %
stock_clearance         100.0 %
discounted_price          0.0 %
total_customer_price      0.0 %
dtype: str

Původní počet sloupců: 10
Počet sloupců po očištění: 9
